In [2]:
import os
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)



# 👇 import the helpers you already defined
from bert_random_search import compute_metrics
from bert_random_search import tokenize_batch_factory



ModuleNotFoundError: No module named 'transcript_preprocessing'

In [ ]:


from transcript_preprocessing import get_stratified_kfold_splits


def run_cv_collect_predictions_bert(
    df: pd.DataFrame,
    transcript_col: str,
    model_name: str,
    max_len: int,
    learning_rate: float,
    num_train_epochs: int,
    batch_size: int,
    weight_decay: float = 0.01,
    warmup_ratio: float = 0.0,
    n_splits: int = 5,
    seed: int = 42,
    output_dir_base: str = "checkpoints_bert_cv",
):
    """
    Run stratified K-fold CV with BERT and collect predictions & per-fold metrics.

    Returns:
        y_true_all, y_pred_all, fold_metrics
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenize_batch = tokenize_batch_factory(transcript_col, tokenizer, max_len)

    all_true: List[np.ndarray] = []
    all_pred: List[np.ndarray] = []
    fold_metrics: List[Dict[str, float]] = []

    num_labels = df["Label"].nunique()

    for fold_idx, train_idx, val_idx in get_stratified_kfold_splits(
        transcript_df=df,
        transcript_col=transcript_col,
        label_col="Label",
        n_splits=n_splits,
        seed=seed,
    ):
        print(f"\n=== BERT CV: Fold {fold_idx} / {n_splits} ({transcript_col}) ===")

        train_df = df.iloc[train_idx].reset_index(drop=True)
        val_df   = df.iloc[val_idx].reset_index(drop=True)

        train_ds = Dataset.from_pandas(train_df)
        val_ds   = Dataset.from_pandas(val_df)

        train_tok = train_ds.map(tokenize_batch, batched=True)
        val_tok   = val_ds.map(tokenize_batch, batched=True)

        # Label -> labels
        train_tok = train_tok.rename_column("Label", "labels")
        val_tok   = val_tok.rename_column("Label", "labels")

        # Drop unused columns if present
        cols_to_remove = [
            "Record-ID",
            "Class",
            "Transcript_PFT",
            "Transcript_CTD",
            "Transcript_SFT",
            "__index_level_0__",
        ]
        cols_to_remove = [c for c in cols_to_remove if c in train_tok.column_names]
        train_tok = train_tok.remove_columns(cols_to_remove)
        val_tok   = val_tok.remove_columns(cols_to_remove)

        train_tok.set_format("torch")
        val_tok.set_format("torch")

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=num_labels,
        )

        fold_output_dir = os.path.join(
            output_dir_base, f"{transcript_col}_fold_{fold_idx}"
        )
        os.makedirs(fold_output_dir, exist_ok=True)

        training_args = TrainingArguments(
            output_dir=fold_output_dir,
            evaluation_strategy="epoch",
            save_strategy="epoch",
            learning_rate=learning_rate,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            num_train_epochs=num_train_epochs,
            weight_decay=weight_decay,
            warmup_ratio=warmup_ratio,
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
            logging_steps=50,
            save_total_limit=2,
            report_to=[],
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_tok,
            eval_dataset=val_tok,
            tokenizer=tokenizer,
            compute_metrics=compute_metrics,  # 👈 imported from bert_random_search
        )

        trainer.train()
        metrics = trainer.evaluate()
        print("Fold metrics:")
        for k, v in metrics.items():
            print(f"  {k}: {v}")

        # Predict on this fold's val set
        preds_output = trainer.predict(val_tok)
        logits = preds_output.predictions
        y_pred = np.argmax(logits, axis=-1)
        y_true = val_tok["labels"].numpy()

        all_true.append(y_true)
        all_pred.append(y_pred)

        fold_metrics.append(
            {
                "fold": fold_idx,
                "accuracy": metrics["eval_accuracy"],
                "macro_f1": metrics["eval_macro_f1"],
                "weighted_f1": metrics["eval_weighted_f1"],
                "precision_macro": metrics["eval_precision_macro"],
                "recall_macro": metrics["eval_recall_macro"],
                "precision_weighted": metrics["eval_precision_weighted"],
                "recall_weighted": metrics["eval_recall_weighted"],
            }
        )

    y_true_all = np.concatenate(all_true)
    y_pred_all = np.concatenate(all_pred)

    return y_true_all, y_pred_all, fold_metrics
